In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.cluster import KMeans

import os
from PIL import Image

In [ ]:
IMG_SIZE = (128, 128)  # MobileNetV2 input — smaller keeps CPU training tractable; GTSRB crops are low-res anyway
BATCH_SIZE = 32
NUM_CLASSES = 43

In [3]:
def get_dataset(directory,type_of_data="train",IMAGE_SIZE=(48,48),num_classes=43):
    X = [] 
    y = [] 
    
    # Iterate through image folders
    for i in range(num_classes):
        path = os.path.join(directory,type_of_data,str(i)) 
        
        # List all the images in the folder
        for j in os.listdir(path):  
            
            # Open the image
            image = Image.open(path + '/'+ j) 
            
            # Resize it to (48,48)
            image = image.resize(IMAGE_SIZE) 
            
            # Convert it to a numpy array for easier use
            image = np.array(image) 
            
            # Add the image and the labels to the lists
            X.append(image) 
            y.append(i)
            
    return np.array(X), np.array(y)

In [4]:
def load_image(path,rgba=False):
    loaded_image = Image.open(path)
    if rgba:
        loaded_image = loaded_image.convert("RGB")
    loaded_image = loaded_image.resize((48,48))
    loaded_image = np.array(loaded_image)
    loaded_image = np.expand_dims(loaded_image,axis=0)
    return loaded_image

In [5]:
def calculate_results(y_true, y_pred):
    # Calculate model accuracy
    model_accuracy = accuracy_score(y_true, y_pred) * 100
    # Calculate model precision, recall and f1 score using "weighted average
    model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
    model_results = {"accuracy": model_accuracy,
                  "precision": model_precision,
                  "recall": model_recall,
                  "f1": model_f1}
    return model_results

In [6]:
def recognize_feature(image_path, model):
    # Read, resize, and match MobileNet dimensions
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    
    # Expand dimensions for a batch format: (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Scale raw pixel intensities down into the exact -1 to 1 model criteria
    processed_img = preprocess_input(img_batch)
    
    # Generate distribution probabilities for both the sign type and color heads
    class_probs, color_probs = model.predict(processed_img)
    class_index = np.argmax(class_probs[0])
    confidence = class_probs[0][class_index]
    color_index = np.argmax(color_probs[0])
    color_confidence = color_probs[0][color_index]
    
    return class_index, confidence, COLOR_NAMES[color_index], color_confidence

In [ ]:
import pandas as pd

directory = "/Users/maliya/Desktop/dissertation/Vista/data/raw/consolidated/gtsrb-german-traffic-sign"
num_classes = 43
sign_names = ["limit_zone_20","limit_zone_30","limit_zone_50","limit_zone_60","limit_zone_70",
              "limit_zone_80","end_of_speed_limit","limit_zone_100","limit_zone_120",
              "no_passing","no_passing_for_trucks","right_of_way","priority_road",
              "yield_right_of_way","stop","prohibited_for_all_vehicles","tractors_and_trucks_prohibited",
              "entry_prohibited","danger","single_curve_left","single_curve_right","double_curve",
              "rough_road","slippery_road","road_narrows","construction_site","signal_lights_ahead","pedestrian_crosswalk_ahead",
              "children","bicycle_crossing","snow_ahead","wild_animal_crossing","end_of_all_restrictions",
               "mandatory_right","mandatory_left","mandatory_ahead","mandatory_ahead_right",
              "mandatory_ahead_left","mandatory_down_right","mandatory_down_left","traffic_circle","end_of_no_passing_zone",
              "end_of_no_passing_zone_trucks"]

# --- Build the training table straight from the GTSRB CSV index ------------------
# The previous version loaded every image into RAM at 48x48 and never trained the
# model, so the exported .mlpackage had random classifier heads. Here we index the
# files, attach the colour label (via Meta.csv) and the ROI box, and stream the
# pixels from disk during training instead.
meta = pd.read_csv(os.path.join(directory, "Meta.csv"))
COLOR_NAMES = ["red", "blue", "yellow", "white_black"]
NUM_COLORS = len(COLOR_NAMES)
class_to_color = meta.set_index("ClassId")["ColorId"].astype(int).to_dict()

train_df = pd.read_csv(os.path.join(directory, "Train.csv"))
train_df["abspath"] = train_df["Path"].apply(lambda p: os.path.join(directory, p))
train_df["color"] = train_df["ClassId"].map(class_to_color).astype(int)
# Normalised ROI box [y1, x1, y2, x2] for tf.image.crop_and_resize
train_df["box_y1"] = train_df["Roi.Y1"] / train_df["Height"]
train_df["box_x1"] = train_df["Roi.X1"] / train_df["Width"]
train_df["box_y2"] = train_df["Roi.Y2"] / train_df["Height"]
train_df["box_x2"] = train_df["Roi.X2"] / train_df["Width"]

train_rows, valid_rows = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df["ClassId"]
)
print(f"{len(train_rows)} train / {len(valid_rows)} val images, "
      f"{train_df['ClassId'].nunique()} classes, {NUM_COLORS} colours")


In [ ]:
import tempfile

AUTOTUNE = tf.data.AUTOTUNE
_cache_root = tempfile.mkdtemp(prefix="gtsrb_cache_")


def make_dataset(rows, *, training, cache_name):
    paths = rows["abspath"].to_numpy()
    boxes = rows[["box_y1", "box_x1", "box_y2", "box_x2"]].to_numpy("float32")
    y_cls = tf.keras.utils.to_categorical(rows["ClassId"].to_numpy(), NUM_CLASSES)
    y_col = tf.keras.utils.to_categorical(rows["color"].to_numpy(), NUM_COLORS)

    ds = tf.data.Dataset.from_tensor_slices((paths, boxes, y_cls, y_col))

    def decode(path, box, yc, yk):
        img = tf.io.decode_png(tf.io.read_file(path), channels=3)
        img = tf.image.crop_and_resize(img[tf.newaxis], box[tf.newaxis], [0], IMG_SIZE)[0]
        img = tf.cast(tf.round(img), tf.uint8)          # cache compact uint8 crops
        return img, yc, yk

    # Decode + ROI-crop once, then reuse the cache on every epoch.
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE).cache(
        os.path.join(_cache_root, cache_name)
    )
    if training:
        ds = ds.shuffle(4096, reshuffle_each_iteration=True)

    def prep(img, yc, yk):
        img = preprocess_input(tf.cast(img, tf.float32))  # MobileNetV2 -> [-1, 1]
        return img, {"sign_class": yc, "sign_color": yk}

    ds = ds.map(prep, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_dataset = make_dataset(train_rows, training=True, cache_name="train")
valid_dataset = make_dataset(valid_rows, training=False, cache_name="valid")


In [ ]:
base_model = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # train the new heads first, fine-tune later

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)          # compress feature-map spatial dims
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)                       # mitigate overfitting

sign_class_output = layers.Dense(NUM_CLASSES, activation="softmax", name="sign_class")(x)
sign_color_output = layers.Dense(NUM_COLORS, activation="softmax", name="sign_color")(x)

model = models.Model(inputs=inputs, outputs=[sign_class_output, sign_color_output])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={"sign_class": "categorical_crossentropy", "sign_color": "categorical_crossentropy"},
    loss_weights={"sign_class": 1.0, "sign_color": 0.3},
    metrics={"sign_class": "accuracy", "sign_color": "accuracy"},
)
model.summary()


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor="val_sign_class_accuracy", mode="max",
                  patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint("mobilenetv2_multitask_best.keras", monitor="val_sign_class_accuracy",
                    mode="max", save_best_only=True, verbose=1),
]

# 1) Train the freshly-added heads with the MobileNetV2 backbone frozen.
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=8,
    callbacks=callbacks,
)

# 2) Light fine-tuning: unfreeze the last block of the backbone at a low LR.
base_model.trainable = True
for layer in base_model.layers[:-24]:
    layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={"sign_class": "categorical_crossentropy", "sign_color": "categorical_crossentropy"},
    loss_weights={"sign_class": 1.0, "sign_color": 0.3},
    metrics={"sign_class": "accuracy", "sign_color": "accuracy"},
)
history_ft = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=4,
    callbacks=callbacks,
)

val_metrics = model.evaluate(valid_dataset, return_dict=True, verbose=0)
print("Validation:", {k: round(v, 4) for k, v in val_metrics.items()})
model.save("mobilenetv2_multitask.keras")


In [ ]:
import coremltools as ct
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

export_path = "MobileNetV2.mlpackage"

# Convert the best checkpoint (EarlyStopping already restored best weights, but be explicit).
best_model = tf.keras.models.load_model("mobilenetv2_multitask_best.keras", compile=False)

# coremltools' TensorFlow front end needs a single, fully-concrete graph with the
# weights inlined as constants. Keras 3's SavedModel export produces a polymorphic
# function the converter rejects ("Only a single concrete function is supported"),
# so we trace the model into one concrete function and freeze it.
input_name = "input_image"

@tf.function(input_signature=[tf.TensorSpec([1, IMG_SIZE[0], IMG_SIZE[1], 3], tf.float32, name=input_name)])
def serving_fn(x):
    return best_model(x, training=False)

frozen_func = convert_variables_to_constants_v2(serving_fn.get_concrete_function())

# The model was trained with MobileNetV2 preprocess_input, i.e. pixels mapped to
# [-1, 1] via x/127.5 - 1. Bake that exact transform into the Core ML image input.
coreml_model = ct.convert(
    [frozen_func],
    source="tensorflow",
    inputs=[
        ct.ImageType(
            name=input_name,
            shape=(1, IMG_SIZE[0], IMG_SIZE[1], 3),
            scale=1 / 127.5,
            bias=[-1.0, -1.0, -1.0],
        )
    ],
    convert_to="mlprogram",
)

# The frozen graph loses the Keras head names ("Identity", "Identity_1"); restore
# them by matching each output's last dimension to the class / colour head sizes.
spec = coreml_model.get_spec()
name_by_dim = {NUM_CLASSES: "sign_class_probs", NUM_COLORS: "sign_color_probs"}
for output in spec.description.output:
    dim = output.type.multiArrayType.shape[-1]
    if dim in name_by_dim and output.name != name_by_dim[dim]:
        ct.utils.rename_feature(spec, output.name, name_by_dim[dim])

coreml_model = ct.models.MLModel(spec, weights_dir=coreml_model.weights_dir)
coreml_model.short_description = "MobileNetV2 multitask GTSRB traffic-sign classifier (sign class + colour)"
coreml_model.input_description[input_name] = f"{IMG_SIZE[0]}x{IMG_SIZE[1]} RGB traffic-sign image (tight crop)"
coreml_model.save(export_path)

print(f"Exported to {os.path.abspath(export_path)}")
print("Inputs :", [i.name for i in coreml_model.get_spec().description.input])
print("Outputs:", [(o.name, tuple(o.type.multiArrayType.shape)) for o in coreml_model.get_spec().description.output])


In [13]:
from pathlib import Path
from PIL import Image

directory = "/Users/maliya/Desktop/dissertation/Vista/data/raw/consolidated/gtsrb-german-traffic-sign"
dataset_dir = Path(directory)

# Loop through all images matching standard formats
extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
image_paths = []

for ext in extensions:
    image_paths.extend(dataset_dir.glob(ext))
loaded_images = []
for path in image_paths:
    class_index, confidence, color_name, color_confidence = recognize_feature(path, model)
    print(class_index, confidence, color_name, color_confidence)
    img = Image.open(path)
    loaded_images.append(img)
    dominant_color(img)
    print(bgr_to_name(img))


In [15]:
def dominant_color(image_bgr, k=3):
    """Return the most common color in an image region as (B, G, R)."""
    # Downsample for speed — exact pixel count doesn't matter for a color average.
    small = cv2.resize(image_bgr, (50, 50), interpolation=cv2.INTER_AREA)
    pixels = small.reshape(-1, 3).astype(np.float32)
    kmeans = KMeans(n_clusters=min(k, len(pixels)), n_init=4, random_state=0)
    labels = kmeans.fit_predict(pixels)
    # The cluster with the most assigned pixels is the dominant color.
    counts = np.bincount(labels)
    dominant = kmeans.cluster_centers_[np.argmax(counts)]
    return tuple(int(c) for c in dominant) 

In [16]:
def bgr_to_name(bgr):
    """Very rough color-name bucketing — swap for a proper color-name lookup
    (e.g. webcolors, or a nearest-neighbor match against a named-color table)
    if you need accurate names rather than raw RGB."""
    b, g, r = bgr
    if max(r, g, b) < 60:
        return "black"
    if min(r, g, b) > 200:
        return "white"
    if r > g and r > b:
        return "red" if r - max(g, b) > 40 else "pink/brown"
    if g > r and g > b:
        return "green"
    if b > r and b > g:
        return "blue"
    if r > 150 and g > 150 and b < 100:
        return "yellow"
    return "gray/mixed"

In [17]:
def identify_signal_state(image_bgr):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    brightness_score = s.astype(np.float32) * v.astype(np.float32)
    threshold = np.percentile(brightness_score, 85)
    lit_mask = brightness_score >= max(threshold, 40 * 40)
    if not np.any(lit_mask):
        return "unknown"
    hue_pixels = h[lit_mask]
    red_mask = (hue_pixels <= 12) | (hue_pixels >= 165)
    orange_mask = (hue_pixels > 12) & (hue_pixels <= 33)
    green_mask = (hue_pixels > 33) & (hue_pixels <= 110)
    counts = {
        "red": int(np.count_nonzero(red_mask)),
        "orange": int(np.count_nonzero(orange_mask)),
        "green": int(np.count_nonzero(green_mask)),
    }
    if max(counts.values()) == 0:
        return "unknown"
    return max(counts, key=counts.get)

